# Correlation spectrum of Gamma for QNN measurement-layer layouts

Inspects the structure-constants matrix Gamma (with Gamma = U @ diag(S) @ V^T) for a fixed re-uploading QNN circuit, across several measurement-layer connectivities (`meas_layer_layout_vec`). For each layout, random Gamma matrices are masked to the QNN's basis-function mask (via `compute_basis_functions_masks_qnns`), and the resulting correlation spectrum S and its purity tr(S^4) are computed and compared against a reference model built from an average-spectrum-matched, unmasked Gamma. All quantities are recomputed directly in this notebook (nothing is loaded from disk).

In [ ]:
# Importing necessary packages
import sys
import importlib

import numpy as np

import matplotlib.pyplot as plt

import scipy


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import analytical_FIM_functions_np
importlib.reload(analytical_FIM_functions_np)

import compute_basis_functions_masks_qnns
importlib.reload(compute_basis_functions_masks_qnns)
import compute_basis_functions_masks_qnns as basis_masks

Define the QNN circuit (`no_qubits`, `qnn_layout`, `ent_layer_layout`) and the set of measurement-layer connectivities to scan (`meas_layer_layout_vec`), plus the local per-feature/per-parameter basis dimensions d, d_tilde and the resulting global dimensions D, K, D_tot.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ----------------------------------- Experiments specs ------------------------------------ ##
### ---------------------------------------------------------------------------------------- ###

# No. of qubits
no_qubits = 4
name_no_qubits = str(no_qubits)

# Layouts of QNNs
qnn_layout = ['inp', 'var', 'inp', 'var']
ent_layer_layout = 0
meas_layer_layout_vec = [0,
                         [(0, 1)],
                         [(0, 1), (1, 2)], 
                         [(0, 1), (2, 3)],
                         [(0, 1), (1, 2), (2, 3)],
                         1,
                         2]
name_meas_layer_layout_vec = ['NONE',
                              '1PAIR',
                              '2PAIR_SEP',
                              '2PAIR_NNS',
                              '3PAIR_NNS',
                              '1PBC',
                              '2PBC']

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 50
no_V_samples_name = str(no_matrix_realiz)

# No. of layers, parameters, features
no_var_layers = 0
no_enc_layers = 0
for ll in qnn_layout:
    if (ll=='var'):
        no_var_layers = no_var_layers + 1
    if (ll=='inp'):
        no_enc_layers = no_enc_layers + 1
no_params = no_qubits * no_var_layers
no_encodings = no_qubits * no_enc_layers
max_freq = no_enc_layers
no_features = no_qubits
name_no_var_layers = str(no_var_layers)
name_no_enc_layers = str(no_enc_layers)
name_no_params = str(no_params)

# Local and global dimensions
d_loc_ins = 2 * max_freq + 1
d_loc_par = 3
name_d_loc_ins = str(d_loc_ins)
name_d_loc_par = str(d_loc_par)
D_ins = d_loc_ins ** no_features
D_pars = d_loc_par ** no_params
D_tot = D_ins * D_pars

Monte-Carlo estimator of the normalized effective dimension d_eff (Eq. 12 of the paper), evaluated as a large-c_n limit of logdet(I + c_n * F_hat). Defined here for completeness but not used later in this notebook.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ---------------------------- Function for effective dimension ---------------------------- ##
### ---------------------------------------------------------------------------------------- ###

def eff_dim_liminf(FIMs):
    cn = 1.0e12
    nsamples = FIMs.shape[0]
    npars = FIMs.shape[1]
    logdets = np.zeros(nsamples)
    for i in range(0,nsamples):
        cnF = cn * FIMs[i,:,:]
        IplusF = np.eye(npars) + cnF
        logdets[i] = np.linalg.slogdet(IplusF)[1]
    effdim = 2.0 * (scipy.special.logsumexp(0.5 * logdets) - np.log(nsamples)) / np.log(cn)
    return effdim

For each measurement layout: compute the basis-function mask over the joint input x parameter basis from the QNN's backward light-cones, draw `no_matrix_realiz` random Gamma matrices restricted to that mask, and record each realization's correlation spectrum S (via SVD Gamma = U diag(S) V^T) and its purity tr(S^4).

In [ ]:
corr_spectra_all = dict()

for nmeas in range(len(meas_layer_layout_vec)):
    meas_layer_layout = meas_layer_layout_vec[nmeas]
    name_meas_layer_layout = name_meas_layer_layout_vec[nmeas]

    BWLC_dict, params_dict, BWLC_ins_dict, inputs_dict = basis_masks.qnn_qubits_backward_lightcones(no_qubits, qnn_layout, 
                                                                                                    ent_layer_layout, meas_layer_layout)

    basis_mask = basis_masks.basis_functions_mask_vector_qnn(no_qubits, BWLC_dict, BWLC_ins_dict, no_params, 
                                                             d_loc_par, no_features, d_loc_ins)

    basis_inputs_mask, basis_params_mask = basis_masks.inputs_params_basis_mask_vectors_qnn(no_qubits, BWLC_dict, BWLC_ins_dict, 
                                                                                            no_params, d_loc_par, no_features, d_loc_ins)
    
    gamma_mask = np.reshape(basis_mask, (D_ins, D_pars))
    nnz_ratio = np.sum(basis_mask) / D_tot

    corr_spectra_layout = []
    rank_spectra_layout = []
    purity_spectra_layout = []
    
    ### Loop over random model draws
    for nm in range(no_matrix_realiz):
        Gamma = 2.0 * np.random.rand(D_ins, D_pars) - 1.0
        Gamma = Gamma * gamma_mask

        ### if matrix is long with large max. dim., SVD is inefficient
        ### and it is convenient to diagonalize G * G.T if A long
        ### G = U S V.T  ==>  G.T = V S U.T  ==>  G*G.T = U S^2 U.T
        ### S V.T = U.T G  ==>  V.T = inv(S) U.T G
        if (D_pars>6000 and D_ins<2000):
            G2 = np.matmul(Gamma, np.transpose(Gamma))
            S2, U = np.linalg.eigh(G2)
            III = np.argsort(S2)
            S2 = np.real(S2[III])
            U = U[:, III]
            ### check if there are eigenvalues very close to 0, if so, remove them
            III = (S2 > 1.0e-08)
            S2 = S2[III]
            U = U[:, III]
            S = np.sqrt(S2)
            ### Get Vt matrix
            Vh = np.matmul(np.matmul(np.diag(1.0/S), np.transpose(U)), Gamma)
        else:
            U, S, Vh = np.linalg.svd(Gamma, full_matrices=True)

        ### Calculate S purity
        S_normalized = S / np.sqrt(np.sum(S**2.0))
        purity_S = np.sum(S_normalized ** 4.0)
        rank_S = np.sum((S_normalized >= 1.0e-10))
        rank_spectra_layout.append(rank_S)
        corr_spectra_layout.append(S_normalized)
        purity_spectra_layout.append(purity_S)

    corr_spectra_layout = np.asarray(corr_spectra_layout)
    purity_spectra_layout = np.asarray(purity_spectra_layout)
    rank_spectra_layout = np.asarray(rank_spectra_layout)

    dict_layout = dict()
    dict_layout['S'] = corr_spectra_layout
    dict_layout['rank_S'] = rank_spectra_layout
    dict_layout['purity_S'] = purity_spectra_layout
    dict_layout['no_in_basis_states'] = np.sum(basis_inputs_mask)
    dict_layout['no_param_basis_states'] = np.sum(basis_params_mask)
    corr_spectra_all[nmeas] = dict_layout
    print('Finished ', nmeas)

Print, per measurement layout, the size of the input/parameter basis-function mask and the mean/std (over random Gamma realizations) of the correlation-spectrum rank and purity tr(S^4).

In [ ]:
for nmeas in range(len(meas_layer_layout_vec)):
    meas_layer_layout = meas_layer_layout_vec[nmeas]
    name_meas_layer_layout = name_meas_layer_layout_vec[nmeas]
    rank_spectra = corr_spectra_all[nmeas]['rank_S']
    no_in_basis_states = corr_spectra_all[nmeas]['no_in_basis_states']
    no_param_basis_states = corr_spectra_all[nmeas]['no_param_basis_states']
    purity_spectra = corr_spectra_all[nmeas]['purity_S']
    print(name_meas_layer_layout)
    print('No. input basis states: ' + str(no_in_basis_states))
    print('No. param. basis states: ' + str(no_param_basis_states))
    print('Rank S: ', np.mean(rank_spectra), ' with std.: ', np.std(rank_spectra))
    print('Purity S: ', np.mean(purity_spectra), ' with std.: ', np.std(purity_spectra))
    print(' ')

Plot the correlation spectrum s_rho vs. eigenvalue index for every random Gamma realization, one color per measurement layout.

In [ ]:
fs = 28
figsize = (8,4)
plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:olive']

for nmeas in range(len(meas_layer_layout_vec)):
    meas_layer_layout = meas_layer_layout_vec[nmeas]
    name_meas_layer_layout = name_meas_layer_layout_vec[nmeas]
    corr_spectra_layout = corr_spectra_all[nmeas]['S']

    y = corr_spectra_layout[0, :]
    x = np.arange(1, len(y) + 1)
    ax.plot(x, y, '-', color=clrs[nmeas], linewidth=4, label=name_meas_layer_layout)
    for n in range(1, corr_spectra_layout.shape[0]):
        y = corr_spectra_layout[n, :]
        x = np.arange(1, len(y) + 1)
        ax.plot(x, y, '-', color=clrs[nmeas], linewidth=4)
        
    #x_c = np.arange(1, len(y_c) + 1)
    #ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
    #ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22, loc='lower left')
ax.legend(fontsize=22)
ax.set_xlabel(r'$\mathrm{eigenvalue\;index}$', fontsize=fs)
ax.set_ylabel(r'$s_{\rho}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')
fig.savefig('corrspectr_layouts.png', bbox_inches='tight')
fig.savefig('corrspectr_layouts.pdf', bbox_inches='tight')

Reference/sanity check: build unmasked random Gamma matrices of each layout's basis-set size, normalized to match that layout's average correlation-spectrum shape, for comparison against the masked-Gamma correlation spectra computed above.

In [ ]:
for nmeas in range(len(meas_layer_layout_vec)):
    meas_layer_layout = meas_layer_layout_vec[nmeas]
    name_meas_layer_layout = name_meas_layer_layout_vec[nmeas]
    rank_spectra = corr_spectra_all[nmeas]['rank_S']
    no_in_basis_states = corr_spectra_all[nmeas]['no_in_basis_states']
    no_param_basis_states = corr_spectra_all[nmeas]['no_param_basis_states']
    corr_spectra_layout = corr_spectra_all[nmeas]['S']
    S = np.mean(corr_spectra_layout, axis=0)
    S_normalized = S / np.sqrt(np.sum(S**2.0))
    
    for nm in range(no_matrix_realiz):
        Gamma = 2.0 * np.random.rand(no_in_basis_states, no_param_basis_states) - 1.0

        ### if matrix is long with large max. dim., SVD is inefficient
        ### and it is convenient to diagonalize G * G.T if A long
        ### G = U S V.T  ==>  G.T = V S U.T  ==>  G*G.T = U S^2 U.T
        ### S V.T = U.T G  ==>  V.T = inv(S) U.T G
        if (D_pars>6000 and D_ins<2000):
            G2 = np.matmul(Gamma, np.transpose(Gamma))
            _, U = np.linalg.eigh(G2)
            ### Get Vt matrix
            Vh = np.matmul(np.matmul(np.diag(1.0/S_normalized), np.transpose(U)), Gamma)
        else:
            U, _, Vh = np.linalg.svd(Gamma, full_matrices=True)